In [28]:
import pandas as pd
import altair as alt

In [29]:
import pandas as pd
disney_df = pd.read_csv('https://query.data.world/s/q6d4wfxrzhpcxwurhr5ly6hv7vb7ti?dws=00000')
disney_df

,movie_title,release_date,genre,MPAA_rating,total_gross,inflation_adjusted_gross
0,Snow White and the Seven Dwarfs,"Dec 21, 1937",Musical,G,"$184,925,485","$5,228,953,251"
1,Pinocchio,"Feb 9, 1940",Adventure,G,"$84,300,000","$2,188,229,052"
2,Fantasia,"Nov 13, 1940",Musical,G,"$83,320,000","$2,187,090,808"
3,Song of the South,"Nov 12, 1946",Adventure,G,"$65,000,000","$1,078,510,579"
4,Cinderella,"Feb 15, 1950",Drama,G,"$85,000,000","$920,608,730"
...,...,...,...,...,...,...
574,The Light Between Oceans,"Sep 2, 2016",Drama,PG-13,"$12,545,979","$12,545,979"
575,Queen of Katwe,"Sep 23, 2016",Drama,PG,"$8,874,389","$8,874,389"
576,Doctor Strange,"Nov 4, 2016",Adventure,PG-13,"$232,532,923","$232,532,923"
577,Moana,"Nov 23, 2016",Adventure,PG,"$246,082,029","$246,082,029"


In [30]:
disney_df['release_date'] = pd.to_datetime(disney_df['release_date'])
disney_df['total_gross'] = (
    disney_df['total_gross']
    .astype(str)
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .astype(float)
)

In [31]:
genres = sorted(disney_df['genre'].dropna().unique().tolist())
genre_dropdown = alt.binding_select(options=[None] + genres, labels=['All Genres'] + genres, name="Select Genre: ")
genre_select = alt.selection_point(fields=['genre'], bind=genre_dropdown)

main_chart = alt.Chart(disney_df).mark_circle(size=100).encode(
    x=alt.X('release_date:T', title='Release Year'),
    y=alt.Y('total_gross:Q', title='Total Gross (USD)', axis=alt.Axis(format='$.2s')),
    color=alt.Color('genre:N', title='Genre'),
    tooltip=['movie_title', 'release_date', 'total_gross', 'genre']
).add_params(
    genre_select
).transform_filter(
    genre_select
).properties(
    width=1000,
    height=700,
    title="The Evolution of Disney Box Office Success"
)

main_chart

alt.Chart(...)

In [32]:
genre_chart = alt.Chart(disney_df).mark_bar().encode(
    x=alt.X('average(total_gross):Q', title='Avg Gross', axis=alt.Axis(format='$.2s')),
    y=alt.Y('genre:N', sort='-x', title='Genre'),
    color=alt.value('orange')
).properties(width=1000, height=1000, title="Which Genres Make the Most?")

genre_chart

alt.Chart(...)

In [33]:
renaissance_df = disney_df[(disney_df['release_date'] >= '1989-01-01') & (disney_df['release_date'] <= '1999-12-31')]

#filter because it doesnt look right with all the movies
top_30_renaissance = renaissance_df.nlargest(30, 'total_gross')

renaissance_chart = alt.Chart(top_30_renaissance).mark_bar().encode(
    x=alt.X('total_gross:Q', title='Total Gross (USD)', axis=alt.Axis(format='$.2s')),
    y=alt.Y('movie_title:N', sort='-x', title=None),
    color=alt.Color('total_gross:Q', scale=alt.Scale(scheme='blues'), legend=None),
    tooltip=['movie_title', 'genre', 'release_date', 'total_gross']
).properties(
    width=1000,
    height=700,
    title="Top 25 Hits of the Disney Renaissance (1989-1999)"
)
renaissance_chart

alt.Chart(...)

In [35]:
# Save files for your github.io repository
main_chart.save('disney_main.json')
genre_chart.save('disney_genre.json')
renaissance_chart.save('renaissance_chart.json')